In [ ]:
import csv
import json
import pandas as pd
import numpy as np
from pydantic import ValidationError,field_validator
from datetime import datetime
import requests

In [ ]:
import os

os.makedirs("data", exist_ok=True)

In [ ]:
def parse_row(row):
    parsed = row.copy()
    Schema = {
        "age":int,
        "fee_inr":int,
        "prior_no_shows":int,
        "distance_km":float
    }
    
    for column,data_type in Schema.items():
        try:
            parsed[column] = data_type(parsed[column])
        except ValueError as e:
            return None,column,e

    return parsed,None,None

    

In [ ]:
with open("data/appointment.csv", newline="") as appointment_csv:
    reader = csv.DictReader(appointment_csv)
    headers = reader.fieldnames    
    print(f"Header: {headers}")
    rejected_rows = []
    active_rows = []
    for row in reader:
        result,column,err = parse_row(row)       
        if result is None:            
            #rejected_rows.append({"row":row,"failed_column":column,"Error":err})
            rejected_rows.append(row)
        else:
            active_rows.append(result)

    if headers and "appointment_id" in headers:
        print(f"Total Rows:{reader.line_num-1}")
    else:
        print(f"Total Rows:{reader.line_num}")

    print(len(rejected_rows))


In [ ]:
with open("data/rejected_rows.json", "w", encoding="utf-8") as f:
  json.dump(rejected_rows, f, indent=2)

In [ ]:
df = pd.read_csv("data/appointment.csv")


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
print("Null counts:\n", df.isnull().sum())

In [ ]:
print("Duplicate count:", df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

In [ ]:
print("After dropping duplicates:", len(df))

In [ ]:
df["booking_channel"] = df["booking_channel"].str.strip().str.title()

In [ ]:
df['appt_date'] = pd.to_datetime(df["appt_date"],errors="coerce")
df['booked_on'] = pd.to_datetime(df["booked_on"],errors="coerce")

In [ ]:
df = df.dropna(subset=["appt_date", "status"])
print("After dropping missing dates/status:", len(df))

In [ ]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")

In [ ]:
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

In [ ]:
median_age = df["age"].median()
print("Calculated Median Age:", median_age)

In [ ]:
median_distance = df["distance_km"].median()
print(median_distance)

In [ ]:
df["age"] = df["age"].fillna(median_age).round().astype(int)


In [ ]:
df["distance_km"] = df["distance_km"].fillna(median_distance)

In [ ]:
for col in ["distance_km", "fee_inr"]:
  q1 = df[col].quantile(0.25)
  q3 = df[col].quantile(0.75)

  print(f"Q1 values in {col}: {q1}")
  print(f"Q3 values in {col}: {q3}")

  iqr = q3 - q1
  lower_fence = q1 - 1.5 * iqr
  upper_fence = q3 + 1.5 * iqr

  print(f"Lower fence in {col}: {lower_fence}")
  print(f"Upper fence in {col}: {upper_fence}")

  capped_count = ((df[col] < lower_fence) | (df[col] > upper_fence)).sum()
  print(f"Capped values in {col}: {capped_count}")
  df[col] = np.clip(df[col], lower_fence, upper_fence)
  

In [ ]:
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

In [ ]:
df["fee_inr"].max()


In [ ]:
df["fee_inr"].min()

In [ ]:
df["distance_km"].max()

In [ ]:
df["distance_km"].min()

In [ ]:
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

In [ ]:
df["no_show"] = df["status"].apply(lambda s: 1 if s == "No-Show" else 0)

In [ ]:
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

In [ ]:
col_means = np.mean(X, axis=0)
print("Column means:", col_means)

In [ ]:
lead_mask = df["lead_days"].to_numpy() > 14
print(
    f"Lead days > 14: count = {lead_mask.sum()}, share ="
    f" {lead_mask.mean():.4f}"
)

In [ ]:
dist_arr = df["distance_km"].to_numpy()
z_scores = (dist_arr - np.mean(dist_arr)) / np.std(dist_arr)
z_count = np.sum(np.abs(z_scores) > 2)
print("Rows with |z| > 2 for distance_km:", z_count)

In [ ]:
clinics_df = pd.read_csv("data/clinics.csv")

In [ ]:
temp_list = []

In [ ]:
for _, row in clinics_df.iterrows():
  lat, lon, name = row["latitude"], row["longitude"], row["clinic_name"]
  temp = None
  url = "https://api.open-meteo.com/v1/forecast"
  params = {
      "latitude": lat,
      "longitude": lon,
      "hourly": "temperature_2m",
      "forecast_days": 1,
  }

  for attempt in range(2):
    try:
      res = requests.get(url, params=params, timeout=10)
      if res.status_code == 200:
        data = res.json()
        if "hourly" in data and "temperature_2m" in data["hourly"]:
          temp = data["hourly"]["temperature_2m"][0]
          break
    except requests.exceptions.RequestException as e:
      if attempt == 1:
        print(f"Clinic {name} failed after retry. Reason: {e}")

  temp_list.append(temp)

clinics_df["temp_c"] = temp_list
clinics_df.to_csv("data/clinics_enriched.csv", index=False)

In [ ]:
df =df.sort_values("appointment_id")

In [ ]:
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

In [ ]:
appointment_cols = [
    "appointment_id",
    "patient_id", 
    "clinic_id", 
    "booked_on", 
    "appt_date", 
    "age", 
    "gender",
    "distance_km", 
    "fee_inr", 
    "booking_channel", 
    "visit_type", 
    "reminder_sent", 
    "prior_no_shows", 
    "status",
    "lead_days", 
    "no_show"
]

df = df[appointment_cols]

In [ ]:
assert df["appointment_id"].nunique() == len(
    df
), "appointment_id is not unique!"
assert df.isnull().sum().sum() == 0, "Missing values found in clean dataset!"

In [ ]:
df.to_csv("data/appointments_clean.csv", index=False)

In [ ]:
with open("data/load_data.sql", "w", encoding="utf-8") as f:
  
  for _, r in clinics_df.iterrows():
    t_val = "NULL" if pd.isna(r["temp_c"]) else r["temp_c"]
    f.write(
        f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}','"
        f"{r['city']}','{r['tier']}',{r['latitude']},{r['longitude']},{t_val});\n"
    )

  
  for _, r in df.iterrows():
    f.write(
        "INSERT INTO appointments VALUES"
        f" ('{r['appointment_id']}','{r['patient_id']}','{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{r['age']},'{r['gender']}',{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}','{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},'{r['status']}',{r['lead_days']},{r['no_show']});\n"
    )


SELECT c.clinic_name,c.city,count(a.appointment_id) as appointments,
sum(a.no_show) AS no_show,
ROUND(100* sum(a.no_show)/count(a.appointment_id),1) AS no_show_pct
FROM clinics c JOIN appointments a on c.Clinic_id=a.Clinic_id group BY c.clinic_id 
HAVING appointments >=95
ORDER BY no_show_pct DESC;

In [ ]:
merged_df = pd.merge(df,clinics_df,on="clinic_id",how="inner")

In [ ]:
clinic_summary = (
    merged_df.groupby(["clinic_name", "city"])
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
    )
    .reset_index()
)

In [ ]:
clinic_summary["no_show_pct"] = round(
    100.0 * clinic_summary["no_shows"] / clinic_summary["appointments"], 1
)
filtered_summary = clinic_summary[clinic_summary["appointments"] >= 95].sort_values(
    by="no_show_pct", ascending=False
)

In [ ]:
print(filtered_summary)

SELECT 
    CASE 
        WHEN lead_days BETWEEN 1 AND 7 THEN '1-7'
        WHEN lead_days BETWEEN 8 AND 14 THEN '8-14'
        WHEN lead_days BETWEEN 15 AND 30 THEN '15-30'
        WHEN lead_days >= 31 THEN '31+'
    END AS lead_time_band,
    COUNT(*) AS appointments,
    SUM(no_show) AS no_shows,
    ROUND(100.0 * SUM(no_show) / COUNT(*), 1) AS no_show_pct
FROM appointments a 
WHERE lead_days > 0
GROUP BY lead_time_band
ORDER BY MIN(lead_days);

In [ ]:
valid_df = df[df["lead_days"] > 0].copy()

In [ ]:
def assign_band(days):
    if 1 <= days <= 7:
        return "1-7"
    elif 8 <= days <=14:
        return "8-14"
    elif 15 <=days<=30:
        return "15-30"
    elif days >=31:
        return "31+"
    return np.nan

In [ ]:
valid_df["lead_time_band"] = valid_df["lead_days"].apply(assign_band)

In [ ]:
band_summary = (
    valid_df.groupby("lead_time_band")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum"),
        min_lead_days=("lead_days", "min"), 
    )
    .reset_index()
)

In [ ]:
band_summary["no_show_pct"] = round(
    100.0 * band_summary["no_shows"] / band_summary["appointments"], 1
)

In [ ]:
band_summary = band_summary.sort_values(by="min_lead_days").drop(
    columns=["min_lead_days"]
)

In [ ]:
band_summary